# PhosphoAtlas Excel EDA

Minimal exploratory notebook for inspecting the PhosphoAtlas kinase-substrate Excel file before using it for background kinase selection.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
excel_path = Path("../data/phosphoAtlas_data/2024_PhosphoAtlas 2.0_updated KSP network_withHTKAMconnections.xlsx")
xls = pd.ExcelFile(excel_path)

print("Excel path:", excel_path)
print("Sheet names:", xls.sheet_names)

In [ ]:
sheet_name = xls.sheet_names[0]
df = pd.read_excel(excel_path, sheet_name=sheet_name)

print("Sheet loaded:", sheet_name)
print("Shape:", df.shape)

In [ ]:
df.head(10)

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes

In [ ]:
summary = {
    "n_rows": len(df),
    "n_columns": df.shape[1],
    "n_unique_kinase_genes": df["KINASE GENE"].nunique(dropna=True),
    "n_unique_kinase_accessions": df["KIN_ACC_ID"].nunique(dropna=True),
    "n_unique_substrate_genes": df["SUBSTRATE GENE"].nunique(dropna=True),
    "n_unique_substrate_accessions": df["SUB_ACC_ID"].nunique(dropna=True),
}
pd.Series(summary)

In [ ]:
duplicate_summary = {
    "full_row_duplicates": int(df.duplicated().sum()),
    "duplicate_kinase_accessions": int(df["KIN_ACC_ID"].duplicated().sum()),
    "duplicate_substrate_accessions": int(df["SUB_ACC_ID"].duplicated().sum()),
    "duplicate_sites_within_kinase_substrate": int(
        df.duplicated(subset=["KIN_ACC_ID", "SUB_ACC_ID", "SUB_MOD_RSD", "SITE_GRP_ID"]).sum()
    ),
}
pd.Series(duplicate_summary)

In [ ]:
df["KIN_ACC_ID"].value_counts().head(20)

In [ ]:
df[["KINASE GENE", "KINASE common name", "KIN_ACC_ID"]].drop_duplicates().head(20)

In [ ]:
df[["KINASE GENE", "KINASE common name", "KIN_ACC_ID"]].drop_duplicates().sort_values(["KINASE GENE", "KIN_ACC_ID"]).reset_index(drop=True)

## Confirming: Are the 418 kinase accessions purely kinases, or kinases + substrates?

In [ ]:
# Unique accessions in each role
kinase_accs = set(df["KIN_ACC_ID"].dropna().unique())
substrate_accs = set(df["SUB_ACC_ID"].dropna().str.split("-").str[0].unique())  # strip isoform suffix

# Overlap: proteins that appear as BOTH kinase and substrate in PA
dual_role = kinase_accs & substrate_accs

print(f"Unique kinase accessions (KIN_ACC_ID):     {len(kinase_accs)}")
print(f"Unique substrate accessions (SUB_ACC_ID):  {len(substrate_accs)}")
print(f"Proteins appearing as BOTH:                {len(dual_role)}")
print(f"Kinases that are NOT substrates:           {len(kinase_accs - substrate_accs)}")
print(f"Substrates that are NOT kinases:           {len(substrate_accs - kinase_accs)}")
print(f"\nTotal unique proteins across all PA roles: {len(kinase_accs | substrate_accs)}")

In [ ]:
# Sample of dual-role proteins (kinases that are also substrates in PA)
# These are biologically interesting — e.g. ABL1 phosphorylates itself
dual_role_genes = (
    df[df["KIN_ACC_ID"].isin(dual_role)][["KINASE GENE", "KIN_ACC_ID"]]
    .drop_duplicates()
    .rename(columns={"KINASE GENE": "gene", "KIN_ACC_ID": "acc_id"})
)
print(f"Sample dual-role kinase/substrates (first 15):")
print(dual_role_genes.head(15).to_string(index=False))